# Import libraries

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import os

In [2]:
import tensorflow as tf
import sklearn.metrics as sk_metrics

print(tf.__version__)
tf.random.set_seed(42)

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


2.11.0


# Load the dataset

In [3]:
train_data = pd.read_csv('/kaggle/input/titanic/train.csv')
train_data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')
test_data.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


---   

## EDA 

In [5]:
def preprocess(df, isTrain=False):
    df = df.drop_duplicates()
    if 'Cabin' in df.columns:
        df = df.drop(['Cabin'], axis=1)
    
    df.fillna({'Age': df['Age'].mean(), 'Fare': df['Fare'].mean(), 'Embarked': "S"}, inplace=True)
    return df

In [6]:
train_data = preprocess(train_data)
test_data = preprocess(test_data)

---

## Feature Engineering

1. Tokenize names and extract titles -> correlates with age & sex -> Might help disentangle the two features for better learning?

In [7]:
def extract_title(tokens):
    return tokens[1].split(" ")[1]

def tokenize(name):
    tokens = name.split(",")
#     print(tokens)
    title = extract_title(tokens)
    return tokens, title

def get_titles(df):
    title_lst = []
    for name in df['Name']:
        name_tokens, title = tokenize(name)
        title_lst.append(title)
    return title_lst

In [8]:
train_titles = get_titles(train_data)
train_data['Title'] = train_titles
train_data.head(5)
# print("train titles:")
# for title in set(train_titles):
#     print(title, "->", train_titles.count(title))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,Mr.
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,Mrs.
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,Miss.
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,Mrs.
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,Mr.


In [9]:
test_titles = get_titles(test_data)
test_data['Title'] = test_titles
test_data.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Title
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,Q,Mr.
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,S,Mrs.
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,Q,Mr.
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,S,Mr.
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,S,Mrs.


**One-Hot Encoding:**

In [10]:
# Notice that features: embarked is not used for training & testing

x_train, y_train = train_data.loc[:, ['Pclass','Sex','Age', 'Parch','SibSp', 'Fare', 'Title']], train_data.loc[:, ['Survived']]
x_test = test_data.loc[:, ['Pclass','Sex','Age', 'Parch', 'SibSp', 'Fare', 'Title']]

print(x_train.shape, y_train.shape, x_test.shape)

(891, 7) (891, 1) (418, 7)


In [11]:
import category_encoders as ce

encoder = ce.OrdinalEncoder(cols=['Sex', 'Title'])

x_train = encoder.fit_transform(x_train)
x_test = encoder.transform(x_test)

In [12]:
x_train, y_train = tf.convert_to_tensor(x_train, dtype=tf.float32), tf.convert_to_tensor(y_train, dtype=tf.float32)
x_test = tf.convert_to_tensor(x_test, dtype=tf.float32) 

#### Random Forest Classifier

In [13]:
import tensorflow_decision_forests as tfdf

#### Auto Hyper-param tuning:

In [14]:
# tuner = tfdf.tuner.RandomSearch(num_trials=100, use_predefined_hps=True)

# tuner.choice("num_trees", [100,200,300,400,500,600,700,800,900,1000])

# local_search_space = tuner.choice("growing_strategy", ["LOCAL"])
# local_search_space.choice("max_depth", [3, 4, 5, 6, 8])

# Build the model

In [15]:
model = tfdf.keras.RandomForestModel(name='rcf_sibsp_m', 
                                     num_trees=500,
                                     max_depth=5
                                    )

history = model.fit(x=x_train, y=y_train, verbose=1)
print(history.history)

Use /tmp/tmpyfguqv21 as temporary training directory
Reading training dataset...
Training dataset read in 0:00:05.777799. Found 891 examples.
Training model...
Model trained in 0:00:00.230634
Compiling model...


[INFO 2023-05-23T04:35:27.967081366+00:00 kernel.cc:1214] Loading model from path /tmp/tmpyfguqv21/model/ with prefix 0b3b1e97ebdf4334
[INFO 2023-05-23T04:35:28.020817679+00:00 decision_forest.cc:661] Model loaded with 500 root(s), 13992 node(s), and 7 input feature(s).
[INFO 2023-05-23T04:35:28.021074044+00:00 abstract_model.cc:1311] Engine "RandomForestOptPred" built
[INFO 2023-05-23T04:35:28.02126348+00:00 kernel.cc:1046] Use fast generic engine


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Model compiled.
{'num_examples': [891], 'accuracy': [0.8271604938271605], 'loss': [1.6339915383870554]}


In [16]:
model.summary()

Model: "rcf_sibsp_m"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
Total params: 1
Trainable params: 0
Non-trainable params: 1
_________________________________________________________________
Type: "RANDOM_FOREST"
Task: CLASSIFICATION
Label: "__LABEL"

Input Features (7):
	data:0.0
	data:0.1
	data:0.2
	data:0.3
	data:0.4
	data:0.5
	data:0.6

No weights

Variable Importance: INV_MEAN_MIN_DEPTH:
    1. "data:0.6"  0.395216 ################
    2. "data:0.5"  0.316351 #########
    3. "data:0.0"  0.313328 ########
    4. "data:0.1"  0.302833 #######
    5. "data:0.2"  0.248110 ###
    6. "data:0.4"  0.225574 #
    7. "data:0.3"  0.211693 

Variable Importance: NUM_AS_ROOT:
    1. "data:0.6" 260.000000 ################
    2. "data:0.1" 168.000000 #########
    3. "data:0.0" 36.000000 
    4. "data:0.5" 36.000000 

Variable Importance: NUM_NODES:
    1. "data:0.5" 2279.000000 ################
    2. "dat

In [17]:
p=model.predict(x_train)
p[p >= 0.5] = 1
p[p  < 0.5] = 0
p = p.flatten()
print(p.shape)
# print(p)

confusion_matrix(y_train,p)
print(classification_report(y_train,p))

28/28 [==============================] - 0s 2ms/step
(891,)


NameError: name 'confusion_matrix' is not defined

In [ ]:
predictions = model.predict(x_test)
print("predictions shape:", predictions.shape)

In [ ]:
predictions[predictions >= 0.5] = 1
predictions[predictions < 0.5] = 0
predictions = predictions.flatten()
print(predictions.shape)

---   

## Submission

In [ ]:
submission_df = pd.DataFrame()
submission_df['PassengerId'] = test_data['PassengerId'].copy()


submission_df['Survived'] = predictions

print(submission_df.head(5))
submission_df['Survived'] = pd.to_numeric(submission_df['Survived'], downcast='integer')

submission_df.to_csv('submission.csv', index=False)